# MOD-05 · NB-01 — Clinical Text Preprocessing
### Health Analytics with Python · Module 05: NLP for Clinical Text
---
**Learning objectives**
- Understand unique challenges of clinical NLP vs general-domain NLP
- Tokenise, sentence-split, and normalise clinical text
- Detect and de-identify PHI using rule-based regex
- Expand clinical abbreviations
- Extract section headers, vital signs and lab values
- Build a reusable end-to-end preprocessing pipeline

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `re`, `nltk`, `pandas`, `numpy`

## 1. Setup & Synthetic Clinical Notes

In [ ]:
import re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

NOTES = [
    {"note_id":"N001","patient_id":"PT-00142","note_type":"Discharge Summary",
     "text":(
      "DISCHARGE SUMMARY\nPatient: John Doe, MRN: 12345678, DOB: 01/15/1956\n"
      "Attending: Dr. Smith, MD\n\nHISTORY OF PRESENT ILLNESS:\n"
      "Mr. Doe is a 67 y/o male with h/o DM2, HTN, and CKD stage 3 who presented to the ED "
      "on 3/12/2023 c/o SOB and LE edema x3 days. Pt denies CP or fever. "
      "T 37.2C, HR 98, BP 142/88 mmHg, RR 18, SpO2 94% on RA.\n\n"
      "MEDICATIONS:\n1. Metformin 1000 mg PO BID\n2. Lisinopril 10 mg PO QD\n"
      "3. Furosemide 40 mg PO QAM\n\nASSESSMENT AND PLAN:\n"
      "1. Acute decompensated HF - IV Lasix 80 mg x2 doses, strict I&O, daily weights\n"
      "2. DM2 - continue home meds, check A1c\n3. HTN - BP well controlled\n\n"
      "LABS: Na 138, K 4.1, Cr 1.8, BUN 32, BNP 1450\n"
      "ECHO: EF 35%, severe LV dysfunction\n\nDISCHARGE CONDITION: Stable\n"
      "FOLLOW-UP: Cardiology in 1 week, PCP in 2 weeks")},
    {"note_id":"N002","patient_id":"PT-00289","note_type":"Progress Note",
     "text":(
      "PROGRESS NOTE - 04/18/2023\nPatient: Jane Smith, 72F, MRN: 87654321\n\n"
      "S: Pt c/o worsening dyspnea and productive cough x4 days. "
      "Denies fever, chills, or hemoptysis. PMH: COPD (GOLD III), HTN, former smoker (40 pack-years), "
      "CAD s/p CABG 2015.\n\n"
      "O: T 38.4C, HR 102, BP 138/82, RR 22, SpO2 88% on 2L NC\n"
      "Lungs: diffuse expiratory wheezes bilaterally, decreased BS at bases\n"
      "CXR: bilateral infiltrates, no frank consolidation\n"
      "ABG: pH 7.32, pCO2 52, pO2 58\n\nA/P:\n"
      "1. COPD exacerbation - start Duoneb nebs q4h, methylprednisolone 125 mg IV q8h\n"
      "2. Possible CAP - Azithromycin 500 mg IV QD, Ceftriaxone 1g IV QD\n"
      "3. Hypoxia - increase O2 to 4L NC, goal SpO2 88-92%")},
    {"note_id":"N003","patient_id":"PT-00451","note_type":"Emergency Note",
     "text":(
      "EMERGENCY DEPARTMENT NOTE\n"
      "55M presents with acute onset chest pain 8/10, radiating to L arm. "
      "No relief with NTG x3. EKG: ST elevation in leads II, III, aVF. "
      "Troponin I: 2.4 ng/mL (elevated). "
      "Pt has h/o HTN, hyperlipidemia, DM2, family hx of MI (father at 58). "
      "BP 168/94, HR 88 bpm, RR 16, SpO2 98% on RA.\n\n"
      "IMPRESSION: STEMI - inferior wall MI\n"
      "PLAN: Activate cath lab. ASA 325 mg, Plavix 600 mg load, Heparin infusion. "
      "Allergies: PCN (rash), sulfa (anaphylaxis)")},
]
df_notes = pd.DataFrame(NOTES)
print(f"Clinical notes: {df_notes.shape}")
for _, r in df_notes.iterrows():
    print(f"  {r.note_id}: {r.note_type} ({len(r.text)} chars)")


## 2. Clinical Challenges Overview

| Challenge | Example | Impact |
|---|---|---|
| **Abbreviations** | SOB, HTN, DM2, BID | Out-of-vocabulary words |
| **Negation** | "denies fever", "no CP" | False positive extraction |
| **Measurements** | BP 142/88, SpO2 94% | Require special regex |
| **Sections** | HPI, A&P, LABS | Document-level semantics |
| **PHI** | Names, MRN, DOB | HIPAA de-identification |

## 3. Clinical Abbreviation Expansion

In [ ]:
CLINICAL_ABBREVS = {
    "DM2":"type 2 diabetes mellitus","HTN":"hypertension","CKD":"chronic kidney disease",
    "HF":"heart failure","CAD":"coronary artery disease",
    "COPD":"chronic obstructive pulmonary disease","PE":"pulmonary embolism",
    "SOB":"shortness of breath","CP":"chest pain","LE":"lower extremity",
    "c/o":"complaining of","h/o":"history of","PMH":"past medical history",
    "s/p":"status post","Pt":"patient","pt":"patient",
    "HR":"heart rate","BP":"blood pressure","RR":"respiratory rate",
    "SpO2":"oxygen saturation","EF":"ejection fraction",
    "BNP":"brain natriuretic peptide","A1c":"hemoglobin A1c",
    "PO":"by mouth","IV":"intravenous","BID":"twice daily","QD":"once daily",
    "QAM":"every morning","q4h":"every 4 hours","q8h":"every 8 hours",
    "EKG":"electrocardiogram","CXR":"chest X-ray","ABG":"arterial blood gas",
    "ECHO":"echocardiogram","NTG":"nitroglycerin","ASA":"aspirin",
    "Cr":"creatinine","BUN":"blood urea nitrogen","NC":"nasal cannula",
}

def expand_abbreviations(text, abbrev_dict=CLINICAL_ABBREVS):
    for abbrev, expansion in sorted(abbrev_dict.items(), key=lambda x: -len(x[0])):
        pattern = r'\b' + re.escape(abbrev) + r'\b'
        text = re.sub(pattern, expansion, text)
    return text

sample = "Pt c/o SOB and LE edema. HTN and DM2 noted. HR 98, BP 142/88, SpO2 94% on RA."
print("Original:", sample)
print("Expanded:", expand_abbreviations(sample))


## 4. PHI Detection & De-identification

In [ ]:
PHI_PATTERNS = {
    "PATIENT_NAME" : r'(?:Patient|Pt)\s*:\s*[A-Z][a-z]+\s+[A-Z][a-z]+',
    "PROVIDER_NAME": r'(?:Dr\.|Attending)\s*:\s*[A-Z][a-z]+\s+[A-Z][a-z]+',
    "MRN"          : r'MRN\s*:\s*\d{6,10}',
    "DATE_OF_BIRTH": r'DOB\s*:\s*\d{1,2}/\d{1,2}/\d{4}',
    "DATE"         : r'\b\d{1,2}/\d{1,2}/\d{4}\b',
    "AGE"          : r'\b\d{2,3}\s*(?:y/o|year[-\s]old|yo)\b',
}

def detect_phi(text):
    found = []
    for phi_type, pattern in PHI_PATTERNS.items():
        for m in re.finditer(pattern, text, re.IGNORECASE):
            found.append({"type":phi_type,"text":m.group(),"start":m.start(),"end":m.end()})
    return sorted(found, key=lambda x: x["start"])

def deidentify(text):
    result = text
    spans = []
    for phi_type, pattern in PHI_PATTERNS.items():
        for m in re.finditer(pattern, result, re.IGNORECASE):
            spans.append((m.start(), m.end(), f"[{phi_type}]"))
    for start, end, replacement in sorted(spans, reverse=True):
        result = result[:start] + replacement + result[end:]
    return result

note1 = df_notes.iloc[0]["text"]
phi_found = detect_phi(note1)
print(f"PHI detected ({len(phi_found)} instances):")
for phi in phi_found:
    print(f"  [{phi['type']:15s}] '{phi['text']}'")
print("\n--- De-identified (first 300 chars) ---")
print(deidentify(note1)[:300])


## 5. Section Extraction

In [ ]:
SECTION_PATTERNS = {
    "HPI"        : r"(?:HISTORY OF PRESENT ILLNESS|HPI)\s*:?",
    "MEDICATIONS": r"MEDICATIONS?\s*:?",
    "ASSESSMENT" : r"(?:ASSESSMENT(?:\s+AND\s+PLAN)?|A/?P|IMPRESSION)\s*:?",
    "LABS"       : r"LABS?\s*:?",
    "PLAN"       : r"PLAN\s*:?",
    "DISCHARGE"  : r"DISCHARGE CONDITION\s*:?",
}

def extract_sections(text):
    headers = []
    for name, pattern in SECTION_PATTERNS.items():
        for m in re.finditer(pattern, text, re.IGNORECASE):
            headers.append((m.start(), m.end(), name))
    headers.sort()
    sections = {}
    for i, (start, end, name) in enumerate(headers):
        next_start = headers[i+1][0] if i+1 < len(headers) else len(text)
        content = re.sub(r'\n+', ' ', text[end:next_start]).strip()
        if content:
            sections[name] = content
    return sections

print("Sections extracted per note:\n")
for _, row in df_notes.iterrows():
    secs = extract_sections(row["text"])
    print(f"{row['note_id']} ({row['note_type']}):")
    for sec, content in secs.items():
        print(f"  [{sec:12s}] {content[:70]}...")
    print()


## 6. Vital Signs & Lab Value Extraction

In [ ]:
VITAL_PATTERNS = {
    "temperature"   : r'T\s*(?:emp)?\s*:?\s*(\d{2,3}\.?\d?)\s*[CF]?',
    "heart_rate"    : r'HR\s*:?\s*(\d{2,3})\s*(?:bpm)?',
    "blood_pressure": r'BP\s*:?\s*(\d{2,3})\s*/\s*(\d{2,3})',
    "resp_rate"     : r'RR\s*:?\s*(\d{1,2})',
    "spo2"          : r'SpO2\s*:?\s*(\d{2,3})\s*%',
}
LAB_PATTERNS = {
    "sodium"        : r'Na\s*(\d{3})',
    "potassium"     : r'K\s*(\d\.\d)',
    "creatinine"    : r'Cr\s*(\d\.\d)',
    "bnp"           : r'BNP\s*(\d{2,5})',
    "troponin_i"    : r'Troponin\s*I\s*:?\s*(\d+\.\d+)',
    "ejection_frac" : r'EF\s*(\d{2})\s*%',
}

def extract_vitals_labs(text):
    results = {}
    for name, pattern in {**VITAL_PATTERNS, **LAB_PATTERNS}.items():
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            if name == "blood_pressure":
                results["systolic"]  = float(m.group(1))
                results["diastolic"] = float(m.group(2))
            else:
                try: results[name] = float(m.group(1))
                except: results[name] = m.group(1)
    return results

print("Extracted vitals & labs:\n")
for _, row in df_notes.iterrows():
    extracted = extract_vitals_labs(row["text"])
    print(f"  {row['note_id']}: {extracted}")


## 7. Negation Detection

In [ ]:
NEGATION_TRIGGERS = [
    r'\bno\b', r'\bnot\b', r'\bdenies?\b', r'\bwithout\b',
    r'\bw/o\b', r'\bnever\b', r'\babsent\b', r'\bnegative\b',
]

def is_negated(text, entity, window=5):
    tokens = text.lower().split()
    entity_lower = entity.lower()
    for i, tok in enumerate(tokens):
        if entity_lower in tok:
            context = " ".join(tokens[max(0,i-window):i])
            for trigger in NEGATION_TRIGGERS:
                if re.search(trigger, context, re.IGNORECASE):
                    return True
    return False

test_cases = [
    ("Pt denies chest pain or fever.", "chest pain"),
    ("Patient has no fever or chills.", "fever"),
    ("Bilateral wheezes are present.", "wheezes"),
    ("No consolidation identified.", "consolidation"),
    ("Chest pain 8/10, radiating to left arm.", "chest pain"),
    ("Hypoxia present on admission.", "hypoxia"),
]
print(f"{'Sentence':55s} {'Entity':15s} {'Negated'}")
print("-"*85)
for sentence, entity in test_cases:
    neg = is_negated(sentence, entity)
    status = "NEGATED" if neg else "Affirmed"
    print(f"  {sentence[:52]:52s} {entity:15s} {status}")


## 8. Full Preprocessing Pipeline

In [ ]:
def preprocess_clinical_note(text, note_id=""):
    result = {"note_id":note_id, "original_length":len(text)}
    result["phi_detected"] = detect_phi(text)
    clean = deidentify(text)
    result["expanded_text"] = expand_abbreviations(clean)
    result["sections"] = extract_sections(text)
    result["vitals_labs"] = extract_vitals_labs(text)
    sentences = []
    for line in result["expanded_text"].split("\n"):
        line = line.strip()
        if len(line.split()) > 5:
            sentences.append(line)
    result["sentences"] = sentences
    result["sentence_count"] = len(sentences)
    return result

print("Running preprocessing pipeline...\n")
for _, row in df_notes.iterrows():
    proc = preprocess_clinical_note(row["text"], note_id=row["note_id"])
    print(f"{row['note_id']}: {proc['sentence_count']} sentences | "
          f"{len(proc['phi_detected'])} PHI | "
          f"{len(proc['sections'])} sections | "
          f"{len(proc['vitals_labs'])} values")


## 9. Medication Extraction Exercise

In [ ]:
MED_PATTERN = (
    r'(\d+\.\s+)?([A-Z][a-zA-Z]+(?:\s+[a-zA-Z]+)?)'
    r'\s+(\d+(?:\.\d+)?\s*(?:mg|mcg|g|mL|units?))'
    r'(?:\s+((?:PO|IV|SQ|IM)\s+(?:BID|QD|TID|QAM|q\d+h)))?'
)

def extract_medications(text):
    meds = []
    for m in re.finditer(MED_PATTERN, text, re.IGNORECASE):
        meds.append({"drug":m.group(2).strip(),
                     "dose":m.group(3).strip(),
                     "route_freq":(m.group(4) or "").strip()})
    return meds

for _, row in df_notes.iterrows():
    meds = extract_medications(row["text"])
    if meds:
        print(f"{row['note_id']}:")
        for med in meds:
            print(f"  {med['drug']:20s} {med['dose']:12s} {med['route_freq']}")


---
**Next → NB-02: Clinical Named Entity Recognition**